# Theme 2: RAG Application, Starter Kit

This starter kit contains a complete, working **Retrieval-Augmented Generation (RAG)** application using **Gemma 4**, Google's open-weight model. Your team's task is to adapt and extend it for the problem statement you selected (2A or 2B).

**How RAG works:**

```
 Document ──► 1. split into chunks ──► 2. convert each chunk into an embedding (a list of numbers)
                                                           │
 Question ──► 3. retrieve the chunks whose embeddings are closest to the question's embedding
                                                           │
              4. give ONLY those chunks to Gemma ──► answer with page citations
```

**Models used (both open-weight):**
- `all-MiniLM-L6-v2` (sentence-transformers) creates the embeddings. It runs inside this notebook.
- `gemma-4-26b-a4b-it` (Google) writes the answers. It is accessed through the free Google AI Studio API.

**No GPU is required.** Use the default Colab runtime (CPU).

### Obtaining a free Google AI Studio API key (one per team)
1. Go to **https://aistudio.google.com/apikey** and sign in with a Google account.
2. Select **Create API key** and copy the key.
3. In Colab, select the **key icon** in the left sidebar (**Secrets**).
4. Select **Add new secret**. Set **Name** to `GOOGLE_API_KEY` and **Value** to your key. Enable **Notebook access**.

> **Important:** do not paste the key directly into a code cell. Anyone the notebook is shared with would be able to see it.
>
> **No key yet:** the notebook still runs. The retrieval stages work normally, and the application shows the passages it found in place of a written answer.

### How to use this notebook
- Run the cells **in order, from top to bottom**, the first time (select a cell and press **Shift + Enter**).
- Comments marked **STUDENT TODO** show the intended places to make changes.
- After changing a cell, re-run that cell and any cells below it that depend on it.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

## Step 0: Installation (run once)

In [7]:
# Installs the required libraries. This takes about one minute.
!pip install -q pypdf sentence-transformers "gradio>=6,<7" google-genai

## Step 1: Settings

These values control how documents are divided, searched and answered. They are the main parameters to experiment with.

In [8]:
# STUDENT TODO: the Gemma 4 model that writes the answers.
#   "gemma-4-26b-a4b-it"  Mixture of Experts, 26B total / about 4B active parameters. Fast. (default)
#   "gemma-4-31b-it"      Dense, 31B parameters. More capable, slower.
GEMMA_MODEL = "gemma-4-26b-a4b-it"

# STUDENT TODO: the number of words in each chunk.
#   Small chunks (50)   more precise matches, but may lose context
#   Large chunks (300)  more context, but less precise and longer prompts
CHUNK_SIZE = 120

# STUDENT TODO: the number of words shared by neighbouring chunks, so sentences are not split.
CHUNK_OVERLAP = 30

# STUDENT TODO: the number of chunks given to Gemma for each question.
TOP_K = 3

# STUDENT TODO: the rules Gemma follows when answering. The rule to use only the context prevents invented answers.
ANSWER_RULES = """You answer questions about a document using ONLY the context passages provided.
- Each passage begins with its page number, for example [Page 2].
- After each fact you use, cite the page in brackets, for example (p. 2).
- If the answer is not in the context, reply exactly: "The answer could not be found in the document."
- Keep answers concise: 2 to 4 sentences."""

TEMPERATURE = 0.2       # a low value keeps answers consistent and close to the document

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # STUDENT TODO (optional): try "BAAI/bge-small-en-v1.5"
print("Settings configured.")

Settings configured.


## Step 2: Connect to Gemma and load the embedding model

In [24]:
import os
from google import genai
from google.genai import types, errors


def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


GOOGLE_API_KEY = _get_secret("GOOGLE_API_KEY")
DEMO_MODE = not GOOGLE_API_KEY
client = None if DEMO_MODE else genai.Client(api_key=GOOGLE_API_KEY)


def _friendly_error(e):
    code = getattr(e, "code", None)
    if code in (400, 401, 403) and "key" in str(e).lower():
        return "Error: the GOOGLE_API_KEY was rejected. Check that the complete key is saved in Colab Secrets, then re-run this cell."
    if code == 429:
        return "Error: the free-tier usage limit has been reached. Wait about one minute and try again."
    if code == 404:
        return f"Error: the model '{GEMMA_MODEL}' was not found. Check the GEMMA_MODEL setting."
    return f"Error: the request to Gemma failed ({e})."


def generate(user_text, system_text=None, temperature=TEMPERATURE, demo_reply="(Demonstration mode response)"):
    """Sends one message to Gemma and returns its reply as text.
    In demonstration mode, returns `demo_reply`. If the request fails, returns a message beginning with "Error:"."""
    if DEMO_MODE:
        return demo_reply
    try:
        response = client.models.generate_content(
            model = GEMMA_MODEL,
            contents = user_text,
            config = types.GenerateContentConfig(
                system_instruction = system_text,
                temperature = temperature,
                thinking_config = types.ThinkingConfig(thinking_level="minimal"),  # faster replies
            ),
        )
        return (response.text or "").strip()
    except errors.APIError as e:
        return _friendly_error(e)
    except Exception as e:
        return f"Error: {e}"


if DEMO_MODE:
    print("Demonstration mode: no GOOGLE_API_KEY was found. Retrieval will work; answers will show the passages found.")
else:
    print(f"Connected to Google AI Studio. Model: {GEMMA_MODEL}")
    print(generate("In one sentence, what is an open-weight AI model?"))

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_NAME)   # runs on the CPU
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")

Connected to Google AI Studio. Model: gemma-4-26b-a4b-it
An open-weight AI model is a machine learning model where the trained parameters (weights) are made publicly available, allowing developers to run, study, and fine-tune the model on their own hardware.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


In [28]:
test_question = "What are the examination timings?"

reply, found = answer(test_question)

print("Question:", test_question)
print("Answer:", reply)

print("\nRetrieved passages:")
for c in found:
    print(f"[Page {c['page']}, similarity {c['score']:.2f}]")
    print(c["text"])

Question: What are the examination timings?
Answer: Tests are conducted for a duration of 60 minutes [Page 31]. Mid Term examinations have a duration of 90 minutes [Page 31]. For UG and PG courses, the duration is 3 hours, though drawing courses require 4 hours [Page 31].

Retrieved passages:
[Page 31, similarity 0.65]
by the subject experts in the question papers. Finally, separate question paper packets shall have to be handed over to the Controller of Examinations/Dean(Examination) or to any other university authority so designated by Vice- Chancellor / Chairperson.  Selection of Question papers for the Examination One among the set of question papers shall be selected randomly by the Controller of Examination/Dean(Examination) or any other university authority so designated by Vice-Chancellor / Chairperson for the examination. ii. Evaluation System a. Tests Tests are conducted centrally by making seating arrangement as done in the semester end examination for 25 marks. The duratio

In [29]:
test_question = "What is the duration of midterm examinations?"

reply, found = answer(test_question)

print("QUESTION:", test_question)
print("\nGENERATED ANSWER:\n", reply)

print("\nRETRIEVED SOURCES:")

for c in found:
    print(f"\nPage: {c['page']}")
    print(f"Similarity: {c['score']:.3f}")
    print(c["text"])

QUESTION: What is the duration of midterm examinations?

GENERATED ANSWER:
 The duration of the Mid Term examination shall be 90 minutes (p. 31).

RETRIEVED SOURCES:

Page: 31
Similarity: 0.762
in the semester end examination for 25 marks. The duration of the test shall be 60minutes. Valuation is carried out by the concerned staff members. b. Mid Term Examination Mid Term examination is conducted centrally by making seating arrangement as done in the semester end examination for 50 marks. The duration of the examination shall be 90 minutes valuation is carried out by the concerned staff members. c.Assignment and Quiz During the course faculty gives assignments and conducts quizzes/surprise tests and each component is evaluated for 5 marks. Minimum of two assignment and quizzes/ surprise tests are conducted for each subject in a semester.

Page: 31
Similarity: 0.611
by the subject experts in the question papers. Finally, separate question paper packets shall have to be handed over to th

In [30]:
test_question = "What is the monthly hostel fee at SSIT Tumkur?"

reply, found = answer(test_question)

print("QUESTION:", test_question)
print("\nANSWER:", reply)

print("\nRETRIEVED SOURCES:")

for c in found:
    print(f"\nPage: {c['page']}")
    print(f"Similarity: {c['score']:.3f}")
    print(c["text"][:500])

QUESTION: What is the monthly hostel fee at SSIT Tumkur?

ANSWER: The answer could not be found in the document.

RETRIEVED SOURCES:

Page: 4
Similarity: 0.358
SRI SIDDHARTHA ACADEMY OF HIGHER EDUCATION (DEEMED TO BE UNIVERSITY) Declared under Section 3 of the UGC Act, 1956, MHRD GOI No. F.9-31/2006-U.3 (A) dated: 30/05/2008 Accredited ‘A’ Grade by NAAC Agalakote, B.H.Road, Tumkur – 572 107.KARNATAKA, INDIA. Ph. 0816- 2275516, 2275512 , 2275514 Fax : 0816-2275510 website: sahetumkur.ac.in email: info@sahe.in No. SSAHE/ACA/06/ 2018 Date: 22.01.2018 NOTIFICATION Sub: Merging of Various Schemes of 1st MBBS UG course with running RS2 scheme. Ref: 1). AC me

Page: 29
Similarity: 0.342
of the Chief Superintendent and other appointed staff shall be separately issued by the Academy. He shall ensure that all the appointed staff is aware of their duties and responsibilities. 1.2. Fee Structure and Schedule: The notification shall include the Fee structure. Dates for payment of examination fees w

In [21]:
print("PDF TEXT:")
print(SAMPLE_PAGES[0][:500])

print("\nKNOWLEDGE BASE TEXT:")
print(knowledge_base["chunks"][0]["text"][:500])

PDF TEXT:
CS 101: Introduction to Programming (Sample Syllabus). Instructor: Dr. A. Rao. Email: arao@example.edu.
    Lectures are held Monday and Wednesday, 10:00 to 11:30 AM, in Room B-204. Labs are on Friday from 2:00 to 4:00 PM
    in the Computer Lab (Room C-110). Office hours: Tuesday 3:00 to 5:00 PM, or by appointment. This course introduces
    programming using Python. No prior coding experience is required.

KNOWLEDGE BASE TEXT:
CS 101: Introduction to Programming (Sample Syllabus). Instructor: Dr. A. Rao. Email: arao@example.edu. Lectures are held Monday and Wednesday, 10:00 to 11:30 AM, in Room B-204. Labs are on Friday from 2:00 to 4:00 PM in the Computer Lab (Room C-110). Office hours: Tuesday 3:00 to 5:00 PM, or by appointment. This course introduces programming using Python. No prior coding experience is required.


In [23]:
from pypdf import PdfReader

# 1. Read the actual SSIT PDF
PDF_PATH = "/content/2.5.4 b. Link for details of examination reforms.pdf"

reader = PdfReader(PDF_PATH)

# 2. Replace the old sample pages
SAMPLE_PAGES = [
    page.extract_text() or ""
    for page in reader.pages
]

# 3. Remove empty pages
SAMPLE_PAGES = [
    page for page in SAMPLE_PAGES
    if page.strip()
]

# 4. Rebuild the knowledge base using the SSIT PDF
print(
    build_knowledge_base(
        SAMPLE_PAGES,
        "SSIT Examination Reforms"
    )
)

# 5. Verify the indexed text
print("\nFIRST INDEXED CHUNK:")
print(knowledge_base["chunks"][0]["text"][:500])

Loaded **SSIT Examination Reforms**: 39 pages, 172 chunks.

FIRST INDEXED CHUNK:
SRI SIDDHARTHA ACADEMY OF HIGHER EDUCATION (DEEMED TO BE UNIVERSITY) Declared under Section 3 of the UGC Act, 1956, MHRD GOI No. F.9-31/2006-U.3 (A) dated: 30/05/2008 Accredited ‘A’ Grade by NAAC Agalakote, B.H.Road, Tumkur – 572 107.KARNATAKA, INDIA. Ph. 0816- 2275516, 2275512 , 2275514 Fax : 0816-2275510 website: sahetumkur.ac.in email: info@sahe.in No. SSAHE/ACA/07/ 2018 Date: 01.02.2018 NOTIFICATION Sub: Conducting supplementary examination of 1st MBBS within two months of announcing results


In [9]:
# STUDENT TODO: replace with text relevant to your problem statement, or upload a PDF in the application instead.
SAMPLE_PAGES = [
    """CS 101: Introduction to Programming (Sample Syllabus). Instructor: Dr. A. Rao. Email: arao@example.edu.
    Lectures are held Monday and Wednesday, 10:00 to 11:30 AM, in Room B-204. Labs are on Friday from 2:00 to 4:00 PM
    in the Computer Lab (Room C-110). Office hours: Tuesday 3:00 to 5:00 PM, or by appointment. This course introduces
    programming using Python. No prior coding experience is required.""",

    """Grading. Weekly lab exercises: 20%. Two mini-projects: 30% (15% each). Midterm exam: 20%. Final exam: 25%.
    Class participation: 5%. A score of 90% or above earns an A, 80 to 89% a B, 70 to 79% a C, and 60 to 69% a D.
    Late submissions lose 10% per day, up to a maximum of 3 days, after which they receive zero. Each student has
    two grace days per semester that may be used to submit any lab late without penalty.""",

    """Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and loops. Week 5: functions.
    Week 6: lists and dictionaries. Week 7: Midterm exam (in class, closed book). Weeks 8-9: files and error handling.
    Weeks 10-11: an introduction to APIs and working with data. Weeks 12-13: mini-project 2, a team project of up to
    3 students. Week 14: review. The final exam is a 2-hour written exam during the examination period.""",

    """Policies. Attendance: students must attend at least 75% of lectures to be eligible for the final exam.
    Academic integrity: students may discuss ideas with classmates, but all submitted code must be their own. AI
    tools may be used for learning and debugging, but their use must be stated in a comment at the top of the file.
    Laptops: bring a laptop to every lab. Students without a laptop may reserve a machine in the Computer Lab.
    Textbook: "Automate the Boring Stuff with Python" (available free online).""",
]
print(f"Sample document contains {len(SAMPLE_PAGES)} pages.")

Sample document contains 4 pages.


## Step 4: The RAG pipeline
Each function is one stage of the pipeline. This is the core of the project and the main code to extend.

In [17]:
import numpy as np
from pypdf import PdfReader

knowledge_base = {"name": None, "chunks": [], "vectors": None}  # populated by build_knowledge_base()


def read_pdf(path):
    """Returns a list of page texts from a PDF file."""
    return [page.extract_text() or "" for page in PdfReader(path).pages]


def split_into_chunks(pages):
    """STAGE 1: split each page into overlapping chunks of about CHUNK_SIZE words."""
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for page_number, text in enumerate(pages, start=1):
        words = text.split()
        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]
            if len(chunk_words) > 5:  # skip very short fragments
                chunks.append({"page": page_number, "text": " ".join(chunk_words)})
            if start + CHUNK_SIZE >= len(words):
                break
    return chunks


def build_knowledge_base(pages, name):
    """STAGE 2: convert each chunk into an embedding."""
    chunks = split_into_chunks(pages)
    if not chunks:
        raise ValueError("No text was found. The PDF may contain scanned images only.")
    vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
    knowledge_base.update(name=name, chunks=chunks, vectors=vectors)
    return f"Loaded **{name}**: {len(pages)} pages, {len(chunks)} chunks."


def retrieve(question, top_k=TOP_K):
    """STAGE 3: find the chunks whose meaning is closest to the question."""
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = knowledge_base["vectors"] @ question_vector   # 1.0 = identical meaning
    best = np.argsort(similarities)[::-1][:top_k]
    return [{**knowledge_base["chunks"][i], "score": float(similarities[i])} for i in best]


def answer(question):
    """STAGE 4: give the retrieved chunks to Gemma and ask it to answer using only them."""
    found = retrieve(question)
    context = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in found)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    demo = (f"**Demonstration mode:** no API key is set, so Gemma did not write an answer. Retrieval completed; "
            f"the most relevant passage is on **page {found[0]['page']}**:\n\n> {found[0]['text'][:400]}...")
    reply = generate(prompt, system_text=ANSWER_RULES, demo_reply=demo)
    return reply, found


# Quick test with the sample document.
# Build the knowledge base using the SSIT examination reforms document
print(
    build_knowledge_base(
        SAMPLE_PAGES,
        "SSIT Examination Reforms"
    )
)

# Test with a question related to the document
test_question = "What examination reforms are mentioned in the document?"

reply, found = answer(test_question)

print(f"\nQuestion: {test_question}")
print(f"Answer: {reply}")

print("\nRetrieved passages:")

for c in found:
    print(
        f"[Page {c['page']}, similarity {c['score']:.2f}] "
        f"{c['text'][:200]}..."
    )


Loaded **SSIT Examination Reforms**: 4 pages, 4 chunks.

Question: What examination reforms are mentioned in the document?
Answer: **Demonstration mode:** no API key is set, so Gemma did not write an answer. Retrieval completed; the most relevant passage is on **page 3**:

> Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and loops. Week 5: functions. Week 6: lists and dictionaries. Week 7: Midterm exam (in class, closed book). Weeks 8-9: files and error handling. Weeks 10-11: an introduction to APIs and working with data. Weeks 12-13: mini-project 2, a team project of up to 3 students. Week 14: review. The final exam is a 2-hour wr...

Retrieved passages:
[Page 3, similarity 0.34] Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and loops. Week 5: functions. Week 6: lists and dictionaries. Week 7: Midterm exam (in class, closed book). Weeks 8-...
[Page 2, similarity 0.31] Grading. Weekly lab exercises: 20%. Two min

In [18]:
print(SAMPLE_PAGES[0][:1000])

CS 101: Introduction to Programming (Sample Syllabus). Instructor: Dr. A. Rao. Email: arao@example.edu.
    Lectures are held Monday and Wednesday, 10:00 to 11:30 AM, in Room B-204. Labs are on Friday from 2:00 to 4:00 PM
    in the Computer Lab (Room C-110). Office hours: Tuesday 3:00 to 5:00 PM, or by appointment. This course introduces
    programming using Python. No prior coding experience is required.


In [19]:
print(
    build_knowledge_base(
        SAMPLE_PAGES,
        "SSIT Examination Reforms"
    )
)

Loaded **SSIT Examination Reforms**: 4 pages, 4 chunks.


## Step 5: Launch the application
Run this cell to open the application below the cell and create a **public link** (`https://....gradio.live`). Re-run it after changing any cell above.

In [12]:
from google.colab import files
uploaded = files.upload()

Saving 2.5.4 b. Link for details of examination reforms.pdf to 2.5.4 b. Link for details of examination reforms.pdf


In [13]:
import os

os.listdir("/content")

['.config',
 '.gradio',
 '2.5.4 b. Link for details of examination reforms.pdf',
 'sample_data']

In [15]:
!pip install pypdf


In [32]:
import gradio as gr
import os

gr.close_all()

APP_TITLE = "SSIT Campus Regulations Assistant"

APP_DESCRIPTION = """
Ask questions about SSIT examination regulations.
Answers are generated using the uploaded document and include page references.
"""

def on_upload(file):
    if file is None:
        return "No file uploaded."

    try:
        return build_knowledge_base(
            read_pdf(file),
            os.path.basename(file)
        )
    except Exception as e:
        raise gr.Error(f"The PDF could not be read: {e}")


def on_ask(question):
    if not question.strip():
        raise gr.Error("Enter a question first.")

    reply, found = answer(question)

    sources = "\n\n".join(
        f"**Page {c['page']}** (similarity {c['score']:.2f})\n> {c['text']}"
        for c in found
    )

    return reply, sources


with gr.Blocks(title=APP_TITLE) as demo:

    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")

    with gr.Row():
        pdf_input = gr.File(
            label="Upload SSIT Regulations PDF",
            file_types=[".pdf"],
            type="filepath"
        )

        with gr.Column():
            status_md = gr.Markdown(
                f"Current document: **{knowledge_base['name']}**"
            )

            sample_btn = gr.Button("Load SSIT Examination Reforms")

    question_box = gr.Textbox(
        label="Ask a question",
        placeholder="For example: What is the duration of midterm examinations?"
    )

    ask_btn = gr.Button("Ask", variant="primary")

    gr.Examples(
        [
            "What is the duration of midterm examinations?",
            "What is the duration of UG examinations?",
            "What is the duration of drawing examinations?",
            "What is the monthly hostel fee at SSIT Tumkur?"
        ],
        inputs=question_box
    )

    answer_md = gr.Markdown()

    with gr.Accordion("Retrieved passages (the text given to Gemma)", open=False):
        sources_md = gr.Markdown()

    pdf_input.upload(
        on_upload,
        inputs=pdf_input,
        outputs=status_md
    )

    sample_btn.click(
        lambda: build_knowledge_base(
            SAMPLE_PAGES,
            "SSIT Examination Reforms"
        ),
        outputs=status_md
    )

    ask_btn.click(
        on_ask,
        inputs=question_box,
        outputs=[answer_md, sources_md]
    )

    question_box.submit(
        on_ask,
        inputs=question_box,
        outputs=[answer_md, sources_md]
    )

demo.launch(share=True)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5a9a6f93a21735f856.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [33]:
evaluation_questions = [
    {
        "question": "What is the duration of midterm examinations?",
        "expected": "90 minutes"
    },
    {
        "question": "What is the duration of UG examinations?",
        "expected": "3 hours"
    },
    {
        "question": "What is the duration of drawing examinations?",
        "expected": "4 hours"
    },
    {
        "question": "What is the monthly hostel fee at SSIT Tumkur?",
        "expected": "Not found in the document"
    },
    {
        "question": "What is the duration of the examination?",
        "expected": "Check the relevant examination type in the document"
    }
]

print("Evaluation questions created:", len(evaluation_questions))

Evaluation questions created: 5


In [34]:
for i, item in enumerate(evaluation_questions, start=1):
    question = item["question"]

    print(f"\n{'=' * 60}")
    print(f"TEST {i}")
    print("Question:", question)

    reply, found = answer(question)

    print("Answer:", reply)

    if found:
        print("Retrieved pages:", [c["page"] for c in found])
    else:
        print("Retrieved pages: None")


TEST 1
Question: What is the duration of midterm examinations?
Answer: The duration of the Mid Term examination shall be 90 minutes (p. 31).
Retrieved pages: [31, 31, 31]

TEST 2
Question: What is the duration of UG examinations?
Answer: The duration of UG examinations is not explicitly stated as a single value, but the document notes that PG courses are 3 hours and drawing courses are 4 hours [Page 31]. Additionally, for internal assessments, the duration of the test is 60 minutes [Page 31].
Retrieved pages: [31, 29, 31]

TEST 3
Question: What is the duration of drawing examinations?
Answer: For drawing courses, the duration of the examination shall be 4 hours (p. 31).
Retrieved pages: [31, 31, 31]

TEST 4
Question: What is the monthly hostel fee at SSIT Tumkur?
Answer: The answer could not be found in the document.
Retrieved pages: [4, 29, 29]

TEST 5
Question: What is the duration of the examination?
Answer: The duration of the examination depends on the type of course and test. Se

In [36]:
for item in results:
    if not item["correct"]:
        print("\nQuestion:", item["question"])
        print("Expected:", item["expected"])
        print("Actual answer:", item["answer"])


Question: What is the monthly hostel fee at SSIT Tumkur?
Expected: Not found in the document
Actual answer: The answer could not be found in the document.

Question: What is the duration of the examination?
Expected: Check the relevant examination type in the document
Actual answer: The duration of the examination depends on the type of course: it is 3 hours for UG courses and 3 hours for PG courses, though drawing courses require 4 hours [Page 31]. Additionally, tests last 60 minutes [Page 31], and Mid Term examinations last 90 minutes [Page 31].


In [37]:
def check_answer(reply, expected):
    reply_lower = reply.lower()

    if expected == "NOT_FOUND":
        refusal_phrases = [
            "could not be found",
            "cannot be found",
            "not mentioned",
            "not provided in the document",
            "does not contain"
        ]
        return any(p in reply_lower for p in refusal_phrases)

    if expected == "EXAM_OVERVIEW":
        required_terms = ["3 hours", "4 hours", "60 minutes", "90 minutes"]
        return all(term in reply_lower for term in required_terms)

    return expected.lower() in reply_lower


# Update the two problematic expected answers
for item in evaluation_questions:
    if item["question"] == "What is the monthly hostel fee at SSIT Tumkur?":
        item["expected"] = "NOT_FOUND"

    elif item["question"] == "What is the duration of the examination?":
        item["expected"] = "EXAM_OVERVIEW"


correct = 0

for item in evaluation_questions:
    reply, found = answer(item["question"])
    item["answer"] = reply
    item["correct"] = check_answer(reply, item["expected"])

    if item["correct"]:
        correct += 1

    print(
        "PASS" if item["correct"] else "CHECK",
        "|", item["question"]
    )

print(f"\nBasic evaluation score: {correct}/{len(evaluation_questions)}")
print(f"Percentage: {correct / len(evaluation_questions) * 100:.1f}%")

PASS | What is the duration of midterm examinations?
PASS | What is the duration of UG examinations?
PASS | What is the duration of drawing examinations?
PASS | What is the monthly hostel fee at SSIT Tumkur?
PASS | What is the duration of the examination?

Basic evaluation score: 5/5
Percentage: 100.0%


In [38]:
for item in evaluation_questions:
    reply, found = answer(item["question"])

    print("\n" + "=" * 70)
    print("QUESTION:", item["question"])
    print("ANSWER:", reply)

    if found:
        print("\nRETRIEVED PASSAGES:")
        for chunk in found:
            print(f"Page: {chunk['page']} | Similarity: {chunk['score']:.3f}")
            print(chunk["text"][:300])
    else:
        print("\nNo passages retrieved.")


QUESTION: What is the duration of midterm examinations?
ANSWER: The duration of the Mid Term examination shall be 90 minutes (p. 31).

RETRIEVED PASSAGES:
Page: 31 | Similarity: 0.762
in the semester end examination for 25 marks. The duration of the test shall be 60minutes. Valuation is carried out by the concerned staff members. b. Mid Term Examination Mid Term examination is conducted centrally by making seating arrangement as done in the semester end examination for 50 marks. 
Page: 31 | Similarity: 0.611
by the subject experts in the question papers. Finally, separate question paper packets shall have to be handed over to the Controller of Examinations/Dean(Examination) or to any other university authority so designated by Vice- Chancellor / Chairperson.  Selection of Question papers for the Examin
Page: 31 | Similarity: 0.561
UG and 3 hours for PG courses. However, for drawing courses the duration of the examination shall be 4 hours. Change in Question paper pattern, if any, sha

In [16]:
from pypdf import PdfReader

pdf_path = "/content/2.5.4 b. Link for details of examination reforms.pdf"

reader = PdfReader(pdf_path)

print("Total pages:", len(reader.pages))

for i, page in enumerate(reader.pages[:3]):
    print(f"\n--- Page {i + 1} ---")    print(page.extract_text()[:1000])

Total pages: 39

--- Page 1 ---
SRI SIDDHARTHA ACADEMY OF HIGHER EDUCATION 
(DEEMED TO BE UNIVERSITY) 
Declared under Section 3 of the UGC Act, 1956, MHRD GOI No. F.9-31/2006-U.3 (A) dated: 30/05/2008 
Accredited ‘A’ Grade by NAAC 
Agalakote, B.H.Road, Tumkur – 572 107.KARNATAKA, INDIA. 
Ph. 0816- 2275516, 2275512 , 2275514  Fax : 0816-2275510 website: sahetumkur.ac.in  email: info@sahe.in 
 
No. SSAHE/ACA/07/ 2018                                  Date: 01.02.2018 
 
NOTIFICATION 
 
Sub: Conducting supplementary examination of 1st  MBBS 
         within two months of  announcing results of regular  
         examination of 1st MBBS  
 
Ref: 1). AC meeting proceedings and its recommendations 
             Dated 18 January 2018     
         2).  Graduate Medical Education regulations- 1997 of  clause 7(7) of  
                              amended up to July 2017 - Medical Council of India 
 
        3). Recommendations of the committee, constituted for the means of  
                  

In [39]:
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)

page_text = reader.pages[30].extract_text() or ""

print(page_text[:6000])

1.18. Malpractice:  
Candidates are prohibited from writing their names and register number, in any place other than indicated. 
Invocation to Gods or any other marks of identification shall not be written anywhere in the answer book. 
Candidates shall not bring into the examination hall any book, portion of book, manuscript or paper of any 
description. They shall not copy or communicate with anyone inside or outside the hall or exchange answer 
books. Candidates shall not be in possession of cell phones, programmable calculators, pen scanners, blue 
tooth equipment or any other equipment which may be used for any kind of malpractice. Candidates 
violating these instructions shall be subjected to physical check and verification by authorized persons. If 
found to be indulging in malpractice the candidate shall be booked for malpractice and sent out of the 
examination hall immediately. The answer book shall be seized, marked as “malpractice case” and signed by 
the Chief of squad/squa

In [40]:
for keyword in ["UG and", "drawing courses", "3 hours", "duration of the examination"]:
    print(f"\n--- Searching for: {keyword} ---")

    pos = page_text.lower().find(keyword.lower())

    if pos != -1:
        print(page_text[max(0, pos - 250):pos + 500])
    else:
        print("Not found on this page.")


--- Searching for: UG and ---
 hall immediately. The answer book shall be seized, marked as “malpractice case” and signed by 
the Chief of squad/squad members and sent to the Dean(Examination). 
 
2.Question Paper Design and evaluation 
i. Question Paper Design 
a.Test 
For both UG and PG courses, the concerned faculty shall set test question paper comprising of five 
questions. Each main question carries five marks. A student has to answer all the five questions.  
b.Mid Term Examination 
For both UG and PG courses, the concerned faculty shall set midterm question paper comprising of five 
questions. The question may consist of subdivisions. Each main question carries ten marks. A student has 
to answer all the five questions.  
c.Semester End Examination 
For UG cours

--- Searching for: drawing courses ---
labus. Questions are shuffled to encourage students to read complete syllabus. A student shall answer 
any 5 questions of his/her choice. 
The duration of the examination shall b

In [41]:
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)

for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""

    if "drawing courses" in text.lower():
        print(f"\nPDF PAGE {i + 1}")
        pos = text.lower().find("drawing courses")
        print(text[max(0, pos - 500):pos + 700])


PDF PAGE 31
0]. The questions shall cover the entire syllabus, unit wise. A 
student has to answer all the questions taking the benefit of internal choice.  
For PG courses, the question paper shall consist of 8 questions (20 marks each) covering the entire 
syllabus. Questions are shuffled to encourage students to read complete syllabus. A student shall answer 
any 5 questions of his/her choice. 
The duration of the examination shall be 1.5 hours to 3 hours for UG and 3 hours for PG courses. 
However, for drawing courses the duration of the examination shall be 4 hours. 
Change in Question paper pattern, if any, shall be brought to the notice of the stakeholders by all means. 
 
 Number of Question Papers to be set in Semester End Examination 
For both UG and PG courses, minimum of two set of question papers shall be set.One by the internal 
faculty and the other by an external faculty from a reputed institution.  
 
 Scrutiny of Question Papers 
Scrutiny of the Question Papers sha

In [42]:
text = reader.pages[30].extract_text() or ""

for keyword in ["drawing courses", "3 hours for PG", "duration of the examination"]:
    start = 0

    while True:
        pos = text.lower().find(keyword.lower(), start)

        if pos == -1:
            break

        print(f"\n--- Found '{keyword}' at position {pos} ---")
        print(text[max(0, pos - 800):pos + 1200])

        start = pos + len(keyword)


--- Found 'drawing courses' at position 2271 ---
n may consist of subdivisions. Each main question carries ten marks. A student has 
to answer all the five questions.  
c.Semester End Examination 
For UG courses, the question paper shall consist of 10 questions (20 marks each) with choice in each unit. 
[ 1 or 2 , 3 or 4, 5 or 6, 7 or 8 and 9 or 10]. The questions shall cover the entire syllabus, unit wise. A 
student has to answer all the questions taking the benefit of internal choice.  
For PG courses, the question paper shall consist of 8 questions (20 marks each) covering the entire 
syllabus. Questions are shuffled to encourage students to read complete syllabus. A student shall answer 
any 5 questions of his/her choice. 
The duration of the examination shall be 1.5 hours to 3 hours for UG and 3 hours for PG courses. 
However, for drawing courses the duration of the examination shall be 4 hours. 
Change in Question paper pattern, if any, shall be brought to the notice of the sta

In [43]:
text = reader.pages[30].extract_text() or ""

print(text[1500:4000])

. Each main question carries ten marks. A student has 
to answer all the five questions.  
c.Semester End Examination 
For UG courses, the question paper shall consist of 10 questions (20 marks each) with choice in each unit. 
[ 1 or 2 , 3 or 4, 5 or 6, 7 or 8 and 9 or 10]. The questions shall cover the entire syllabus, unit wise. A 
student has to answer all the questions taking the benefit of internal choice.  
For PG courses, the question paper shall consist of 8 questions (20 marks each) covering the entire 
syllabus. Questions are shuffled to encourage students to read complete syllabus. A student shall answer 
any 5 questions of his/her choice. 
The duration of the examination shall be 1.5 hours to 3 hours for UG and 3 hours for PG courses. 
However, for drawing courses the duration of the examination shall be 4 hours. 
Change in Question paper pattern, if any, shall be brought to the notice of the stakeholders by all means. 
 
 Number of Question Papers to be set in Semester En

In [44]:
for chunk in knowledge_base["chunks"]:
    if chunk["page"] == 31 and (
        "drawing courses" in chunk["text"].lower()
        or "3 hours for pg" in chunk["text"].lower()
    ):
        print("=" * 60)
        print(chunk["text"])

or 2 , 3 or 4, 5 or 6, 7 or 8 and 9 or 10]. The questions shall cover the entire syllabus, unit wise. A student has to answer all the questions taking the benefit of internal choice. For PG courses, the question paper shall consist of 8 questions (20 marks each) covering the entire syllabus. Questions are shuffled to encourage students to read complete syllabus. A student shall answer any 5 questions of his/her choice. The duration of the examination shall be 1.5 hours to 3 hours for UG and 3 hours for PG courses. However, for drawing courses the duration of the examination shall be 4 hours. Change in Question paper pattern, if any, shall be brought
UG and 3 hours for PG courses. However, for drawing courses the duration of the examination shall be 4 hours. Change in Question paper pattern, if any, shall be brought to the notice of the stakeholders by all means.  Number of Question Papers to be set in Semester End Examination For both UG and PG courses, minimum of two set of question 

In [45]:
for item in evaluation_questions:
    if item["question"] in [
        "What is the duration of UG examinations?",
        "How long are UG examinations?",
        "What is the duration of the UG examination?"
    ]:
        item["expected"] = "1.5 hours to 3 hours"

print("Updated UG examination expectations.")

Updated UG examination expectations.


In [46]:
def check_answer(reply, expected):
    reply_lower = reply.lower()

    if expected == "NOT_FOUND":
        return any(p in reply_lower for p in [
            "could not be found",
            "cannot be found",
            "not mentioned",
            "not provided in the document",
            "does not contain"
        ])

    if expected == "EXAM_OVERVIEW":
        required_terms = ["4 hours", "60 minutes", "90 minutes"]
        return all(term in reply_lower for term in required_terms)

    if expected == "1.5 hours to 3 hours":
        return (
            "1.5 hours" in reply_lower
            and "3 hours" in reply_lower
        )

    return expected.lower() in reply_lower

print("Evaluation checker updated.")

Evaluation checker updated.


## Next steps
1. Re-read your selected problem statement (2A or 2B) in the Theme 2 document.
2. Replace the sample document and example questions with material for your problem statement.
3. Adapt `ANSWER_RULES`, the settings and the application to meet the requirements.
4. Measure the quality of the answers (see "Expected Deliverables" in the problem statement).
5. Attempt items from the **Good to have** section of the problem statement for additional credit.

**Pointers for common extensions:**
- **Markdown and text files (needed for 2B):** read them with `open(path, encoding="utf-8").read()` and treat each file (or each section under a heading) as one page, storing the file name for citations. Allow them in the application with `file_types=[".pdf", ".md", ".txt"]`.
- **Multiple documents:** use `gr.File(file_count="multiple")`, combine the pages of every file, and store the file name in each chunk for citations.
- **Answer without RAG (for comparison):** call `generate(question)` with no context and show the result next to the RAG answer.
- **Relevance threshold:** in `answer()`, if `found[0]["score"]` is below a chosen value, return "The answer could not be found in the document." without calling Gemma.
- **Conversation memory:** replace the question box with `gr.ChatInterface` and include recent messages in the prompt.

## Troubleshooting

| Issue | Resolution |
|---|---|
| The notebook remains in demonstration mode | Add `GOOGLE_API_KEY` in Secrets, enable **Notebook access**, and re-run the Step 2 cell. |
| "API key was rejected" | Create a new key at aistudio.google.com/apikey and update the secret. |
| "Usage limit reached" | Wait about one minute. Free keys allow a limited number of requests per minute. Team members may each create a key. |
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| "No text was found" | The PDF contains scanned images only. Use a text-based PDF. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |